# Autark Python API - Jupyter Integration Test

This notebook tests the `_repr_html_()` integration for displaying Autark specs in Jupyter.

In [1]:
# Setup: Add parent directory to path
import sys
from pathlib import Path

# Add python/ directory to path
python_dir = Path.cwd().parent
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import autark as ak

## Test 1: Simple GeoJSON Map

Load a GeoJSON file and display it on a map.

In [2]:
# Create a simple GeoJSON map spec
neighborhoods = ak.GeoJSON(
    "neighborhoods",
    url="/examples/data/neighborhoods.geojson",
    coordinate_format="EPSG:4326",
    layer_type="polygons"
)

spec1 = ak.Spec(
    metadata=ak.Metadata(
        title="Simple GeoJSON Map",
        description="Basic test of GeoJSON rendering"
    ),
    workspace=ak.Workspace(name="test_geojson", coordinate_format="EPSG:4326"),
    data=[neighborhoods],
    views=[
        ak.Map(
            name="simple_map",
            camera=ak.Camera(pitch=0, bearing=0, zoom=12),
            layers=[
                ak.Layer(neighborhoods, id="neighborhoods_layer", type="polygons")
                    .style(color="#4a90e2", opacity=0.7)
            ]
        )
    ]
)

# Display - this should trigger _repr_html_()
spec1

AutarkSpec(views=[Map(layers=[Layer(source=GeoJSON(name='neighborhoods', url='/examples/data/neighborhoods.geojson', values=None, layer_type='polygons', coordinate_format='EPSG:4326', type='geojson'), id='neighborhoods_layer', type='polygons', encoding=None, style_values={'color': '#4a90e2', 'opacity': 0.7})], name='simple_map', camera=Camera(pitch=0, bearing=0, zoom=12), style=None, type='map')], data=[GeoJSON(name='neighborhoods', url='/examples/data/neighborhoods.geojson', values=None, layer_type='polygons', coordinate_format='EPSG:4326', type='geojson')], transforms=None, links=None, layout=None, metadata=Metadata(title='Simple GeoJSON Map', description='Basic test of GeoJSON rendering', authors=None, created=None), workspace=Workspace(name='test_geojson', coordinate_format='EPSG:4326'))

## Test 2: CSV Points with Histogram

Load CSV data with lat/lng and create a histogram.

In [8]:
trees = ak.CSV(
    "trees",
    url="/examples/data/trees.csv",
    geometry=ak.latlng("latitude", "longitude", coordinate_format="EPSG:4326")
)

spec2 = ak.Spec(
    metadata=ak.Metadata(
        title="CSV Points with Histogram",
        description="Test CSV loading and histogram rendering"
    ),
    workspace=ak.Workspace(name="test_csv", coordinate_format="EPSG:4326"),
    data=[trees],
    views=[
        ak.Map(
            name="points_map",
            camera=ak.Camera(pitch=0, bearing=0, zoom=12),
            layers=[
                ak.Layer(trees, id="trees_layer", type="points")
                    .style(color="#228B22", size=4, opacity=0.6)
            ]
        ),
        ak.Histogram(
            name="tree_histogram",
            source=trees,
            x="latitude",
            bins=30
        )
    ],
    layout=ak.Layout(type="vertical")
)

spec2

AutarkSpec(views=[Map(layers=[Layer(source=CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv'), id='trees_layer', type='points', encoding=None, style_values={'color': '#228B22', 'size': 4, 'opacity': 0.6})], name='points_map', camera=Camera(pitch=0, bearing=0, zoom=12), style=None, type='map'), Histogram(source=CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv'), x='latitude', name='tree_histogram', y=None, bins=30, selection=None, type='histogram')], data=[CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv')], transforms=None, links=None, layout=Layout(type='vertical', columns=None, gap=None), metadata=Metadata(title='CSV Points with Histogram', description='Test CSV loading and histogram rendering', authors=None, created=None), workspace=Workspace(name='test_csv', coordinate_format='EPSG:4326'))

## Test 3: Full Spatial Join Example

Complete workflow with spatial join, thematic mapping, and linked selection.

In [9]:
# Import the working spatial join example
from spatial_join import build_spec

spec3 = build_spec()
spec3

AutarkSpec(views=[Map(layers=[Layer(source='neighborhoods', id='neighborhoods_layer', type='polygons', encoding=Encoding(color=Field(field='properties.sjoin.count.trees', scale=Scale(type='quantile', scheme='greens', domain=None, domain_strategy='minMax')), opacity=None, size=None, height=None), style_values={'opacity': 0.8, 'strokeColor': '#333333', 'strokeWidth': 1}), Layer(source=CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv'), id='trees_layer', type='points', encoding=None, style_values={'color': '#228B22', 'size': 3, 'opacity': 0.6})], name='neighborhood_map', camera=Camera(pitch=0, bearing=0, zoom=12), style=None, type='map'), Histogram(source=GeoJSON(name='neighborhoods', url='/examples/data/neighborhoods.geojson', values=None, layer_type='polygons', coordinate_format='EPSG:4326', type='geojson'), x='sjoin.count.trees', name='tree_count_histogram', y=None, bins=20, selection=Selection(name='tree_count_brush', type='interval', fields=None), type='histogram')], data=[GeoJSON(name='neighborhoods', url='/examples/data/neighborhoods.geojson', values=None, layer_type='polygons', coordinate_format='EPSG:4326', type='geojson'), CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv')], transforms=[SpatialJoin(root=GeoJSON(name='neighborhoods', url='/examples/data/neighborhoods.geojson', values=None, layer_type='polygons', coordinate_format='EPSG:4326', type='geojson'), join=CSV(name='trees', url='/examples/data/trees.csv', values=None, delimiter=None, geometry=LatLngGeometry(latitude='latitude', longitude='longitude', coordinate_format='EPSG:4326', type='latlng'), type='csv'), near=None, group_by=[Aggregation(column='*', op='count', as_=None, normalize=None), Aggregation(column='species', op='collect', as_='tree_species', normalize=None)], type='spatialJoin')], links=[Link(selection=Selection(name='tree_count_brush', type='interval', fields=None), target='neighborhoods_layer', action='highlight')], layout=Layout(type='vertical', columns=None, gap=None), metadata=Metadata(title='Spatial Join - Trees per Neighborhood', description='Demonstrates spatial join between neighborhoods and tree points, aggregating tree counts and visualizing on a thematic map.', authors=None, created='2026-06-07'), workspace=Workspace(name='spatial_join_demo', coordinate_format='EPSG:4326'))

## Test 4: Check JSON Output

Verify the spec serializes correctly to JSON.

In [ ]:
# Print the JSON to verify structure
import json
print(json.dumps(spec1.to_dict(), indent=2))

## Test 5: Export to HTML

Test the `save_html()` method.

In [10]:
# Save to HTML file
output_path = Path("/tmp/autark_test.html")
spec1.save_html(output_path)
print(f"HTML saved to: {output_path}")
print(f"File exists: {output_path.exists()}")
print(f"File size: {output_path.stat().st_size} bytes")

HTML saved to: /tmp/autark_test.html
File exists: True
File size: 2662 bytes


## Expected Results

For each spec display above, you should see:

1. **Interactive visualization** - A rendered map/plot using the TypeScript runtime
2. **Collapsible spec details** - A `<details>` element showing the JSON spec
3. **No console errors** - Check browser console for any JavaScript errors

### Known Issues to Watch For:

1. **Runtime URL** - Default is `/autk-runtime/dist/autk-runtime.js`. May need to adjust based on how you're serving the runtime.
2. **Data URLs** - Relative paths like `/examples/data/...` need proper server setup.
3. **CORS issues** - If loading from local files without a server.

### Debugging Tips:

If the visualization doesn't appear:
- Check the browser console for errors
- Inspect the HTML to verify the script tag is present
- Verify the runtime module URL is accessible
- Check that data file URLs are accessible